*Copyright (c) 2026 Mapenzi Supaki*

Shield: [![CC BY-NC 4.0][cc-by-nc-shield]][cc-by-nc]

This work is licensed under a
[Creative Commons Attribution-NonCommercial 4.0 International License][cc-by-nc].

[![CC BY-NC 4.0][cc-by-nc-image]][cc-by-nc]

[cc-by-nc]: https://creativecommons.org/licenses/by-nc/4.0/
[cc-by-nc-image]: https://licensebuttons.net/l/by-nc/4.0/88x31.png
[cc-by-nc-shield]: https://img.shields.io/badge/License-CC%20BY--NC%204.0-lightgrey.svg

## **Stage 1: Setup**

**Import libraries and set display options**

In [ ]:
# import data analysis libraries
import pandas as pd
import numpy as np

# plotting libraries for later EDA
import matplotlib.pyplot as plt
import seaborn as sns

# make notebook output easier to read
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

import io
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## **Stage 2: Data Loading & Initial Inspection**

Load all datasets from provided CSV strings. Each dataset serves a specific purpose in the analysis:
- fintech_users: User demographic and acquisition information
- fintech_devices: Device characteristics for technical friction analysis
- network_logs: Network conditions during onboarding
- sessions: Session timing and duration context
- kyc_events: Core funnel events with status and error codes

In [ ]:
# load all source files
users = pd.read_csv('fintech_users.csv')
devices = pd.read_csv('fintech_devices.csv')
network = pd.read_csv('network_logs.csv')
sessions = pd.read_csv('sessions.csv')
kyc_events = pd.read_csv('kyc_events.csv')

In [ ]:
# Show columns and first few rows for each table
for name, df in {
    "users": users,
    "devices": devices,
    "network": network,
    "sessions": sessions,
    "events": kyc_events
}.items():
    print(f"\n=== {name.upper()} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3))

## **Stage 3: Data Audit & Quality Checks**

Comprehensive data audit to understand dataset structure, quality, and relationships.
This is crucial for identifying potential issues before analysis.

In [ ]:
def audit_table(df, name):
    print(f"\n--- AUDIT: {name} ---")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

    print("Data Types:\n", df.dtypes)

    print("\nMissing values:\n", df.isna().sum())

    print("\nDuplicates:", df.duplicated().any())

    print("\nUnique values (for object columns):")
    for col in df.columns:
        if df[col].dtype == 'object':
            print(f"  {col}: {df[col].nunique()} unique values")

    # For numeric cols
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    if len(num_cols) > 0:
        print("\nNumeric Summary:\n", df[num_cols].describe())



# Run audit on each table
audit_table(users, "users")
audit_table(devices, "devices")
audit_table(network, "network")
audit_table(sessions, "sessions")
audit_table(kyc_events, "events")

Overall, the data is very clean. The audit_table function successfully ran for all five dataframes (users, devices, network, sessions, and kyc_events), providing a detailed overview of each dataset's structure and quality:

- For each table, it shows the number of rows and columns, along with its memory usage.
- It lists the data type for each column, which is essential for understanding how the data is stored.
- It identifies any missing values, column by column. The only column with missing values is error_code in the kyc_events table, which is expected as errors don't occur in all events.
- It confirms that there are no duplicate rows across any of the tables.
- For object-type columns, it shows the count of unique values, which helps in identifying categorical data and potential inconsistencies.
- For numerical columns, it provides descriptive statistics such as count, mean, standard deviation, and quartile ranges, giving insights into the data distribution.

In [ ]:
# Cross-table consistency checks
# sets of unique user_id values from the dataframes
users_list = set(users['user_id'].unique())
devices_list = set(devices['user_id'].unique())
sessions_list = set(sessions['user_id'].unique())
kyc_list = set(kyc_events['user_id'].unique())
network_list = set(network['user_id'].unique())

# print the total number of unique users found in each of the tables
print(f"\nUsers in users table: {len(users_list)}")
print(f"Users in devices table: {len(devices_list)}")
print(f"Users in sessions table: {len(sessions_list)}")
print(f"Users in kyc_events table: {len(kyc_list)}")
print(f"Users in network_logs table: {len(network_list)}")

# Check if all users in core tables are consistent
core_users = users_list & devices_list & sessions_list & kyc_list
print(f"\nUsers present in all core tables (users, devices, sessions, kyc_events): {len(core_users)}")
print(f"These are: {sorted(core_users)}")



This cross-table consistency checks, shows how many users are present in all the 'core' tables. This ensures there is data integrity for analysis as it involves combined data sources. The check also prints the count of these consistent users and a sorts a list of their IDs.


In [ ]:
# Check session_id consistency

sessions_from_kyc = set(kyc_events['session_id'].unique())
sessions_from_sessions = set(sessions['session_id'].unique())
print(f"\nUnique session_id in kyc_events: {len(sessions_from_kyc)}")
print(f"Unique session_id in sessions: {len(sessions_from_sessions)}")
print(f"All session_ids match: {sessions_from_kyc == sessions_from_sessions}")

# Check for orphaned records
users_without_kyc = users_list - kyc_list
print(f"\nUsers with no KYC events: {len(users_without_kyc)}")
if users_without_kyc:
    print(f"  These users exist but never started KYC: {users_without_kyc}")

# Check whether event user_ids all exist in users
missing_event_users = set(kyc_events["user_id"]) - set(users["user_id"])
print("\nUsers in events not found in users:", len(missing_event_users))

# Check whether event session_ids all exist in sessions
missing_event_sessions = set(kyc_events["session_id"]) - set(sessions["session_id"])
print("\nSessions in events not found in sessions:", len(missing_event_sessions))


This check shows unique session_id values from both the kyc_events and sessions dataframes and compares them to ensure all session IDs align between the two tables.  It then checks for user_ids that exist in the users table but do not have any corresponding records in the kyc_events table. This helps identify users who signed up but never initiated the KYC process.

The analysis depends on these relationships being valid:
- user_id should connect users, devices, network, sessions, and events
- session_id should connect sessions and events

If keys are broken, funnel metrics become misleading.

Summary:
- all event users exist in the users table
- all event sessions exist in the sessions table
- keys are unique where they should be

## **Stage 4: Data Preparation & Feature Engineering**

1. Parse dates and create time-based features
2. Create segmentation buckets for analysis
3. Join all datasets into a single analytical table
4. Handle the network_logs data (multiple rows per user)

**Parse dates**

In [ ]:
# Parse datetimes
users['signup_time'] = pd.to_datetime(users['signup_time'])
sessions['session_start'] = pd.to_datetime(sessions['session_start'])

users['signup_date'] = users['signup_time'].dt.date

**Create derived features**
- age bands
- risk buckets
- network buckets
- camera quality buckets
- day of week
- corrected time-of-day

In [ ]:
# Derive a more reliable time-of-day from session_start
def derive_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

sessions["derived_time_of_day"] = sessions["session_start"].dt.hour.map(derive_time_of_day)
sessions["day_of_week"] = sessions["session_start"].dt.day_name()

The original time_of_day field in sessions.csv does not align well with the actual session_start timestamp, so it is safer to use derived_time_of_day.

In [ ]:
# Filter network_logs to only include our core users
core_users_list = list(core_users)
network_logs_filtered = network[network['user_id'].isin(core_users_list)].copy()
print(f"\nFiltered network_logs from {len(network)} to {len(network_logs_filtered)} rows for core users")

In [ ]:
# Age bands
users["age_band"] = pd.cut(
    users["age"],
    bins=[17, 24, 34, 44, 54, 64, 120],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
)

# Fraud risk buckets
users["fraud_risk_bucket"] = pd.cut(
    users["fraud_risk_score"],
    bins=[-0.01, 0.2, 0.5, 0.8, 1.0],
    labels=["low", "medium", "high", "very_high"]
)

# Camera quality buckets
devices["camera_quality_bucket"] = pd.cut(
    devices["camera_quality_score"],
    bins=[-1, 3, 6, 10],
    labels=["low", "medium", "high"]
)

# Latency buckets
network["latency_bucket"] = pd.cut(
    network["latency_ms"],
    bins=[-1, 100, 200, 1000],
    labels=["low_latency", "medium_latency", "high_latency"]
)

# Upload speed buckets
network["upload_speed_bucket"] = pd.cut(
    network["upload_speed_mbps"],
    bins=[-1, 5, 15, 1000],
    labels=["slow", "moderate", "fast"]
)

# Preview derived fields
print(users[["age", "age_band", "fraud_risk_score", "fraud_risk_bucket"]].head())
print(devices[["camera_quality_score", "camera_quality_bucket"]].head())
print(network[["latency_ms", "latency_bucket", "upload_speed_mbps", "upload_speed_bucket"]].head())
print(sessions[["session_start", "derived_time_of_day", "day_of_week"]].head())

# **Stage 5: Exploratory Data Analysis (EDA)**

In [ ]:
# User distribution by key attributes
print("\n--- User Demographics ---")
print("\nBy Country:")
print(users['country'].value_counts())

print("\nBy Acquisition Source:")
print(users['acquisition_source'].value_counts())

print("\nBy Age Band:")
users['age_band'] = pd.cut(users['age'], bins=[0, 25, 35, 45, 100],
                               labels=['18-25', '26-35', '36-45', '45+'], right=False)
print(users['age_band'].value_counts().sort_index())

print("\n--- Device Distribution ---")
print("\nBy Device Type:")
print(devices['device_type'].value_counts())

print("\nBy OS:")
print(devices['os'].value_counts())

print("\nBy Camera Quality Bucket:")
devices['camera_quality_bucket'] = pd.cut(
    devices['camera_quality_score'],
    bins=[0, 40, 70, 100],
    labels=['Low', 'Medium', 'High'],
    right=False
)
print(devices['camera_quality_bucket'].value_counts())

print("\n--- Network Conditions ---")
print("\nBy Network Type (for core users):")
print(network['network_type'].value_counts())

print("\nLatency distribution (ms):")
print(network['latency_ms'].describe())

print("\nUpload speed distribution (Mbps):")
print(network['upload_speed_mbps'].describe())

### **Standardize the KYC step order**

In [ ]:
# Define the intended funnel order
funnel_steps = [
    "start_kyc",
    "phone_verification",
    "personal_information",
    "document_upload",
    "document_validation",
    "selfie_capture",
    "face_match",
    "address_verification",
    "manual_review",
    "kyc_approved"
]

# Convert to ordered categorical for correct sorting and analysis
kyc_events["kyc_step"] = pd.Categorical(
    kyc_events["kyc_step"],
    categories=funnel_steps,
    ordered=True
)

# Check unique steps
print(kyc_events["kyc_step"].value_counts(dropna=False).sort_index())

If you skip this step, pandas may sort funnel steps alphabetically, which would break your analysis.

#### **Create base joined event table**
We want one row per KYC event, enriched with:

- session context
- user demographics
- device characteristics
- network conditions

This becomes the main table for EDA and root-cause analysis.

In [ ]:
# Join events with session context first
base_events = (
    kyc_events
    .merge(
        sessions[["session_id", "user_id", "session_start", "derived_time_of_day", "day_of_week"]],
        on=["session_id", "user_id"],
        how="left"
    )
    .merge(users, on="user_id", how="left")
    .merge(devices, on="user_id", how="left")
    .merge(network, on="user_id", how="left")
)

print(base_events.shape)
print(base_events.head())

In [ ]:
# Enforce step order: sort by user, session, event_id
base_events = base_events.sort_values(['user_id', 'session_id', 'event_id'])

In [ ]:
print("\nEvent Status Distribution:")
status_counts = base_events['status'].value_counts()
for status, count in status_counts.items():
    pct = (count / len(base_events)) * 100
    print(f"  {status}: {count} ({pct:.1f}%)")

print("\nEvents by Step:")
step_counts = base_events['kyc_step'].value_counts()
for step, count in step_counts.items():
    print(f"  {step}: {count}")

print("\nError Code Analysis:")
error_counts = base_events[base_events['error_code'].notna()]['error_code'].value_counts()
for error, count in error_counts.items():
    print(f"  {error}: {count}")

EDA to help understand:
- population mix
- event distribution
- status distribution
- whether any steps dominate failures or abandonment

#### **Distribution of KYC events by step**

In [ ]:
step_counts = base_events["kyc_step"].value_counts().sort_index()
print(step_counts)

step_counts.plot(kind="bar", figsize=(10, 5), title="KYC Event Volume by Step")
plt.xlabel("KYC Step")
plt.ylabel("Number of Events")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### **Status Distribution**

In [ ]:
status_counts = base_events["status"].value_counts()
print(status_counts)

status_counts.plot(kind="bar", figsize=(6, 4), title="Event Status Distribution")
plt.xlabel("Status")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

#### **Duration Distribution**

In [ ]:
print(base_events["duration_seconds"].describe())

base_events["duration_seconds"].plot(
    kind="hist",
    bins=40,
    figsize=(8, 4),
    title="Distribution of Event Duration"
)
plt.xlabel("Duration (seconds)")
plt.tight_layout()
plt.show()

#### **Retry Count Distribution**

In [ ]:
print(base_events["retry_count"].describe())

base_events["retry_count"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(8, 4),
    title="Retry Count Distribution"
)
plt.xlabel("Retry Count")
plt.ylabel("Event Count")
plt.tight_layout()
plt.show()

A few interesting patterns stood out:
- durations are fairly similar across steps
- retry counts are also fairly uniform

This suggests that the dataset is strongest for funnel and segmentation analysis. Also duration/retry may still be useful, but should not be overinterpreted.

#### **Packet Loss Distribution**

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='network_type', y='packet_loss', data=base_events)
plt.title('Packet Loss Distribution by Network Type')
plt.xlabel('Network Type')
plt.ylabel('Packet Loss')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

This violin plot illustrates the distribution of `packet_loss` for each `network_type`. The width of each violin indicates the density of data points at different packet loss values. The plot compares the typical packet loss rates, their variability, and the presence of any extreme values across various network types.

#### **Upload Speed Distribution**

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='network_type', y='upload_speed_mbps', data=base_events)
plt.title('Upload Speed Distribution by Network Type')
plt.xlabel('Network Type')
plt.ylabel('Upload Speed (Mbps)')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

This violin plot illustrates the distribution of `upload_speed_mbps` for each `network_type`. The width of each violin indicates the density of data points at different upload speeds. The plot helps compare the typical upload speed rates, their variability, and the presence of any extreme values across various network types.

Based on the violin plots for 'packet_loss' and 'upload_speed_mbps' across different 'network_type' categories, we can draw some combined insights:
- Network types like '4G' and 'wifi' tend to show broader distributions for both metrics, indicating a wider range of experienced speeds and packet loss. This might suggest more variability in performance depending on specific conditions or location within these networks.
- '3G' Network: The '3G' network likely exhibits a more concentrated distribution for both packet loss and upload speed, potentially indicating generally lower speeds and less stable connections (higher packet loss) compared to more modern networks. The violin for 3G upload speed would probably be skewed towards lower values, and its packet loss might be higher or more variable.
- '5G' Network: '5G' networks, on the other hand, would typically show distributions favoring higher upload speeds and lower packet loss, reflecting their advanced technology and improved performance. Their violins would likely be concentrated at the higher end for upload speed and lower end for packet loss, with possibly tighter distributions.

It's important to consider that network types with higher average upload speeds (like '5G') might also exhibit lower packet loss, contributing to an overall smoother user experience. Conversely, networks with lower speeds might also struggle with higher packet loss. However, it's also possible to have good speeds with occasional high packet loss, or vice-versa, depending on network congestion and infrastructure.


In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='network_type', y='latency_ms', data=base_events)
plt.title('Latency Distribution by Network Type')
plt.xlabel('Network Type')
plt.ylabel('Latency (ms)')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

This violin plot illustrates the distribution of `latency_ms` for each `network_type`. The width of each violin indicates the density of data points at different latency values. We can see the comparison of the typical latency, spread, and presence of outliers for each network type at a glance.

In [ ]:
plt.figure(figsize=(10, 6))
base_events['latency_ms'].plot(
    kind='hist',
    bins=50, # Using 50 bins for a detailed view
    title='Distribution of Latency (ms)',
    xlabel='Latency (ms)',
    ylabel='Frequency'
)
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

#### **Visualize the KYC event status by step**
Display the proportion of 'success', 'fail', and 'abandon' statuses for each KYC step.



In [ ]:
status_by_step = base_events.groupby(['kyc_step', 'status']).size().unstack(fill_value=0)
status_proportions = status_by_step.div(status_by_step.sum(axis=1), axis=0)
print(status_proportions.head())

In [ ]:
status_proportions.plot(kind='bar', stacked=True, figsize=(12, 7), cmap='viridis')
plt.title('KYC Event Status Proportion by Step')
plt.xlabel('KYC Step')
plt.ylabel('Proportion of Events')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

The visualization reveals that the proportions of 'success', 'fail', and 'abandon' statuses are relatively consistent across most KYC steps, suggesting that underlying issues might be systemic rather than step-specific. It also shows that the 'success' proportion is dominant for most steps.

Note: The steps with a higher absolute number of events could still represent a significant volume of failures or abandonments, even if their proportional rates are not drastically different.

**Key findings:**
*   The proportions of 'success', 'fail', and 'abandon' statuses for KYC events remain relatively consistent across most KYC steps. This suggests that the underlying issues causing failures or abandonment might not be specific to a single step, but rather systemic or related to user behavior patterns throughout the entire KYC journey.
*   The 'success' proportion is dominant for the majority of KYC steps, indicating that a majority of events within each step are completed successfully.
*   Despite proportional consistency, steps with a larger absolute number of events (as seen in the earlier `step_counts` output) might still represent a higher volume of failures or abandonments in absolute terms, even if their *proportion* isn't drastically different. Focusing on these high-volume steps for improvement could yield the most impact.

**Insights:**
*   Prioritize improving high-volume KYC steps, as even a small proportional increase in success in these steps can significantly reduce overall failures and abandonments.
*   Deep-dive into specific error codes at each 'fail' step to pinpoint the exact reasons for failure (e.g., 'blurry\_document' or 'face\_mismatch') and identify targeted solutions.

**Next Steps for Further Analysis:**

1.  **Absolute Failure/Abandonment Counts**: While proportions are useful, it's crucial to also look at the absolute number of failures and abandonments at each step. A small proportional increase in a high-volume step could mean a significant number of lost users.
2.  **Error Code Analysis per Step**: Investigate which `error_code` types are most prevalent at each `fail` step. This will help pinpoint the specific reasons for failure (e.g., 'blurry_document' in `document_upload` or `document_validation`, 'face_mismatch' in `face_match`).
3.  **User Drop-off Analysis**: Analyze the `user_id`s that `abandon` at specific steps. Are certain user segments more prone to abandoning? (e.g., users from certain countries, specific age bands, or those with particular device/network conditions).
4.  **Duration Impact**: Correlate event duration with status. Do longer durations lead to higher abandonment or failure rates at certain steps?
5.  **Multivariate Analysis**: Explore interactions between various user, device, and network attributes with failure/abandonment rates at each step. For example, is there a higher failure rate for users with 'low' camera quality in `selfie_capture` or `face_match` steps?





The chart shows the proportions of 'success', 'fail', and 'abandon' statuses across each KYC step. From this chart, we can observe:

*   **Relatively Consistent Proportions**: It appears that the proportions of success, failure, and abandonment remain quite consistent across most KYC steps. This suggests that the underlying issues causing failures or abandonment might not be specific to a single step, but rather systemic or related to user behavior patterns throughout the entire KYC journey.
*   **Higher Overall Success Rate**: For most steps, the 'success' proportion is dominant, indicating that a majority of events within each step are completed successfully.
*   **Identify Critical Steps**: While proportions are consistent, steps with a larger absolute number of events (as seen in the earlier `step_counts` output) might still represent a higher volume of failures or abandonments in absolute terms, even if their *proportion* isn't drastically different. Focusing on these high-volume steps for improvement could yield the most impact.

**Next Steps for Further Analysis:**

1.  **Absolute Failure/Abandonment Counts**: While proportions are useful, it's crucial to also look at the absolute number of failures and abandonments at each step. A small proportional increase in a high-volume step could mean a significant number of lost users.
2.  **Error Code Analysis per Step**: Investigate which `error_code` types are most prevalent at each `fail` step. This will help pinpoint the specific reasons for failure (e.g., 'blurry_document' in `document_upload` or `document_validation`, 'face_mismatch' in `face_match`).
3.  **User Drop-off Analysis**: Analyze the `user_id`s that `abandon` at specific steps. Are certain user segments more prone to abandoning? (e.g., users from certain countries, specific age bands, or those with particular device/network conditions).
4.  **Duration Impact**: Correlate event duration with status. Do longer durations lead to higher abandonment or failure rates at certain steps?
5.  **Multivariate Analysis**: Explore interactions between various user, device, and network attributes with failure/abandonment rates at each step. For example, is there a higher failure rate for users with 'low' camera quality in `selfie_capture` or `face_match` steps?

**Calculate Absolute Failure and Abandonment Counts**

In [ ]:
filtered_events = base_events[base_events['status'].isin(['fail', 'abandon'])]
absolute_status_counts = filtered_events.groupby(['kyc_step', 'status']).size().unstack(fill_value=0)
print("Absolute counts of 'fail' and 'abandon' statuses by KYC step:")
print(absolute_status_counts.head())

In [ ]:
absolute_status_counts.plot(kind='bar', stacked=True, figsize=(12, 7), cmap='plasma')
plt.title('Absolute Number of Fail and Abandon Events by KYC Step')
plt.xlabel('KYC Step')
plt.ylabel('Number of Events')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

- **Absolute Failure/Abandonment Counts**: `start_kyc` and `phone_verification` steps exhibit the highest absolute numbers of both 'fail' and 'abandon' events. Specifically, `start_kyc` had 19,703 'fail' events and 13,138 'abandon' events.
- While proportional analysis highlights the *rate* of failure or abandonment at each step, the absolute counts identify the *volume* of affected users. For example, a step might have a high failure rate but a low absolute number if few users reach it. Conversely, a step like `start_kyc` might have a moderate rate but a very high absolute number, indicating a significant bottleneck in the overall process.

**Practical implications for improving the KYC process:**
- Focus intervention efforts on high-volume steps like `start_kyc` and `phone_verification`, as improvements there would impact the largest number of users.


**Most prevalent error codes for 'fail' events by KYC step**

In [ ]:
failed_events_with_errors = base_events[base_events['status'] == 'fail'].dropna(subset=['error_code'])
error_counts_by_step = failed_events_with_errors.groupby(['kyc_step', 'error_code']).size().unstack(fill_value=0)

print("Most prevalent error codes for 'fail' events by KYC step:")
print(error_counts_by_step.head())

In [ ]:
error_counts_by_step.plot(kind='bar', stacked=True, figsize=(14, 8), cmap='tab20')
plt.title('Prevalent Error Codes for Fail Events by KYC Step')
plt.xlabel('KYC Step')
plt.ylabel('Number of Fail Events')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Error Code', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

- **Error Code Analysis**: The most prevalent error codes for 'fail' events are `blurry_document`, `document_rejected`, `face_mismatch`, and `network_timeout`. These error types appear across all KYC steps that generate error codes, with their absolute numbers decreasing as users progress through the KYC flow.
- The error code analysis provides granular reasons for these failures, complementing both proportional and absolute analyses by pinpointing *why* users fail at specific steps. This moves beyond just knowing *where* issues occur to understanding *what* the issues are.

**Practical implications for improving the KYC proces:**
- Address specific error codes such as `blurry_document`, `document_rejected`, `face_mismatch`, and `network_timeout` through targeted solutions (e.g., improved instructions, better document scanning technology, enhanced network stability, or clearer face capture guidelines).

**Summary:**
*   The `start_kyc` and `phone_verification` steps account for the highest absolute numbers of both 'fail' and 'abandon' events. The `start_kyc` step alone recorded 19,703 'fail' events and 13,138 'abandon' events.
*   The most prevalent error codes contributing to 'fail' events across all relevant KYC steps are `blurry_document`, `document_rejected`, `face_mismatch`, and `network_timeout`.
*   Although these error types appear consistently across steps, their absolute numbers decrease as users advance through the KYC process.
*   'Fail' events generally outnumber 'abandon' events across most KYC steps.

**Insights / Next Steps:**
*   Prioritize improving the user experience and technical robustness at the `start_kyc` and `phone_verification` steps, as these represent significant bottlenecks impacting a large volume of users.
*   Implement targeted solutions to mitigate common error codes, such as improving image capture guidance for `blurry_document` and `face_mismatch` errors, or enhancing network stability checks for `network_timeout` issues.


# **Stage 6: Funnel Analysis - Core Metrics**